# ELO-Enhanced March Madness Predictions

This notebook implements the primary prediction pipeline:
1. ELO ratings calculated from historical game results
2. XGBoost model trained to predict **ELO residuals** (not raw outcomes)
3. All-matchup predictions generated for Kaggle submission

**Change `CURRENT_SEASON` at the top of each year before running.**

Outputs:
- `output/{CURRENT_SEASON}/elo_enhanced/M/submission.csv`
- `output/{CURRENT_SEASON}/elo_enhanced/W/submission.csv`
- `output/{CURRENT_SEASON}/elo_enhanced/submission_combined.csv`

In [ ]:
# =============================================================================
# CONFIGURATION — Update these each year
# =============================================================================
CURRENT_SEASON = 2026
DATA_DIR = f"../data/{CURRENT_SEASON}"
OUTPUT_DIR = "../output"

# ELO Tuning: set True on first run to find best hyperparameters (takes ~30 min)
PERFORM_TUNING = False

# ELO hyperparameters (from prior tuning runs)
ELO_K_FACTOR = 30
ELO_RECENCY_FACTOR = 1.0
ELO_RECENCY_WINDOW = 15
ELO_CARRY_OVER = 0.75

# Training range for XGBoost residual model
TRAIN_START = 2010
TRAIN_END = CURRENT_SEASON - 1

# Prediction method: 'elo', 'elo_enhanced'
PREDICTION_METHOD = "elo_enhanced"
# =============================================================================

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

from src.data_classes.processing.DataManager import MarchMadnessDataManager
from src.data_classes.processing.EloRatingSystem import EloRatingSystem
from src.data_classes.processing.TeamStatsCalculator import TeamStatsCalculator
from src.data_classes.processing.MLModel import MarchMadnessMLModel
from src.data_classes.processing.Predictor import MarchMadnessPredictor

def make_output_path(year, method, gender, filename):
    path = f"../output/{year}/{method}/{gender}"
    os.makedirs(path, exist_ok=True)
    return f"{path}/{filename}"

os.makedirs("../output", exist_ok=True)
print(f"Season: {CURRENT_SEASON}  |  Data: {DATA_DIR}")

## Men's Tournament

In [ ]:
# --- Initialize men's pipeline ---
# Note: EloRatingSystem takes only data_manager in __init__;
# ELO hyperparameters are passed to calculate_elo_ratings() below.
dm_m = MarchMadnessDataManager(DATA_DIR, gender="M", current_season=CURRENT_SEASON)
dm_m.load_data()

elo_m = EloRatingSystem(dm_m)
stats_m = TeamStatsCalculator(dm_m)
ml_m = MarchMadnessMLModel(dm_m, elo_m, stats_m)

predictor_m = MarchMadnessPredictor(
    data_manager=dm_m,
    elo_system=elo_m,
    stats_calculator=stats_m,
    ml_model=ml_m,
    current_season=CURRENT_SEASON,
)
print("Men's predictor initialized.")

In [ ]:
# Optional: tune ELO hyperparameters via grid search on historical seasons
# This takes ~30 minutes. Run once, then paste best params into config above.
if PERFORM_TUNING:
    print("Tuning ELO parameters for men's tournament...")
    tuning_results = predictor_m.tune_elo_parameters(
        test_seasons=list(range(2019, CURRENT_SEASON)),
        visualize=False,
    )
    print(tuning_results.head(5))
else:
    print("Skipping ELO tuning (PERFORM_TUNING=False)")

In [ ]:
# Calculate ELO ratings with configured hyperparameters
print("Calculating ELO ratings...")
elo_m.calculate_elo_ratings(
    start_year=2003,
    k_factor=ELO_K_FACTOR,
    recency_factor=ELO_RECENCY_FACTOR,
    recency_window=ELO_RECENCY_WINDOW,
    carry_over_factor=ELO_CARRY_OVER,
)
print("ELO ratings calculated.")

# Calculate advanced team statistics
print("Calculating team statistics...")
stats_m.calculate_advanced_team_stats()
print("Stats calculated.")

# Train XGBoost residual model
print("Training XGBoost residual model...")
ml_m.train_model(model_type="xgboost")
print("Model trained.")

In [ ]:
# Backtest on recent historical seasons
print("Backtesting men's model...")
backtest_seasons = [s for s in range(max(2019, TRAIN_START), CURRENT_SEASON) if s != 2020]
backtest_results_m = predictor_m.backtest_multiple_seasons(
    seasons=backtest_seasons,
    method="elo_enhanced",
    visualize=False,
)
if backtest_results_m:
    print(f"Avg Brier score: {backtest_results_m['aggregate']['brier_score']:.4f}")
    print(f"Avg Accuracy:    {backtest_results_m['aggregate']['accuracy']:.4f}")

In [ ]:
# Generate all-matchup predictions for men's
output_m = make_output_path(CURRENT_SEASON, "elo_enhanced", "M", "submission.csv")
sub_m = predictor_m.generate_predictions(
    submission_file=output_m,
    method=PREDICTION_METHOD,
    get_all_matchups=True,
)
print(f"Men's predictions: {len(sub_m)} rows -> {output_m}")

## Women's Tournament

In [ ]:
# --- Initialize women's pipeline ---
dm_w = MarchMadnessDataManager(DATA_DIR, gender="W", current_season=CURRENT_SEASON)
dm_w.load_data()

elo_w = EloRatingSystem(dm_w)
stats_w = TeamStatsCalculator(dm_w)
ml_w = MarchMadnessMLModel(dm_w, elo_w, stats_w)

predictor_w = MarchMadnessPredictor(
    data_manager=dm_w,
    elo_system=elo_w,
    stats_calculator=stats_w,
    ml_model=ml_w,
    current_season=CURRENT_SEASON,
)

print("Calculating women's ELO ratings...")
elo_w.calculate_elo_ratings(
    start_year=2003,
    k_factor=ELO_K_FACTOR,
    recency_factor=ELO_RECENCY_FACTOR,
    recency_window=ELO_RECENCY_WINDOW,
    carry_over_factor=ELO_CARRY_OVER,
)
stats_w.calculate_advanced_team_stats()
ml_w.train_model(model_type="xgboost")
print("Women's models ready.")

In [ ]:
# Generate all-matchup predictions for women's
output_w = make_output_path(CURRENT_SEASON, "elo_enhanced", "W", "submission.csv")
sub_w = predictor_w.generate_predictions(
    submission_file=output_w,
    method=PREDICTION_METHOD,
    get_all_matchups=True,
)
print(f"Women's predictions: {len(sub_w)} rows -> {output_w}")

## Combine into Final Submission

In [ ]:
combined = pd.concat([sub_m[['ID','Pred']], sub_w[['ID','Pred']]], ignore_index=True)
combined_path = f"../output/{CURRENT_SEASON}/elo_enhanced/submission_combined.csv"
os.makedirs(os.path.dirname(combined_path), exist_ok=True)
combined.to_csv(combined_path, index=False)

print(f"Combined submission: {len(combined)} rows")
print(f"  Men's:   {len(sub_m)} matchups")
print(f"  Women's: {len(sub_w)} matchups")
print(f"  Saved:   {combined_path}")
combined.head()

## Bracket Visualization

Once tournament seeds are announced, run this cell to generate a bracket image.

In [ ]:
# Bracket visualization (requires seeds to be available in data)
try:
    from src.visualization.bracket_viz import visualize_bracket

    bracket_path = make_output_path(CURRENT_SEASON, "elo_enhanced", "M", "bracket.png")
    fig = visualize_bracket(
        predictor=predictor_m,
        season=CURRENT_SEASON,
        method=PREDICTION_METHOD,
        output_path=bracket_path,
    )
    plt.show()
    print(f"Bracket saved to {bracket_path}")
except Exception as e:
    print(f"Bracket visualization failed (seeds may not be available yet): {e}")